# 04 - LIME Analysis: Local Interpretable Model-agnostic Explanations

## SHAP vs LIME — At a Glance
| Dimension | SHAP | LIME |
|-----------|------|------|
| Scope | Global + Local | Local only |
| Approach | Game-theory (Shapley values) | Local linear approximation |
| Speed | Fast for tree models | Slower (perturbation-based) |
| Model-agnostic | No (optimised versions per model type) | Yes |
| Consistency | High | Can vary with random seed |

**Key Question:** For a specific patient, which features contributed most to the model's prediction — and in which direction?

In [0]:
import pickle
import numpy as np
import pandas as pd
import lime
import lime.lime_tabular
import matplotlib.pyplot as plt

# Load preprocessed data and models
with open('/tmp/preprocessed_data.pkl', 'rb') as f:
    data = pickle.load(f)

with open('/tmp/models.pkl', 'rb') as f:
    models = pickle.load(f)

X_train = data['X_train']
X_test  = data['X_test']
y_test  = data['y_test']
feature_names = data['feature_names']
best_model = models['xgb']

# Initialise LIME explainer
explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train,
    feature_names=feature_names,
    class_names=['No Disease', 'Disease'],
    mode='classification',
    random_state=42
)

print("LIME Tabular Explainer initialised.")
print(f"Training data shape: {X_train.shape}")
print(f"Feature names: {feature_names}")

In [0]:
# Explain two individual predictions with LIME

# Patient 1: High-risk (predicted as Disease)
high_risk_idx = int(np.where(best_model.predict(X_test) == 1)[0][0])
exp_high = explainer.explain_instance(
    data_row=X_test[high_risk_idx],
    predict_fn=best_model.predict_proba,
    num_features=10
)

print(f"\nPatient {high_risk_idx} — LIME Explanation (Predicted: Disease):")
print(f"Prediction probability: {best_model.predict_proba(X_test[high_risk_idx:high_risk_idx+1])}")
for feat, weight in exp_high.as_list():
    print(f"  {feat}: {weight:.4f}")

fig = exp_high.as_pyplot_figure()
plt.title(f"LIME Explanation — High Risk Patient (idx {high_risk_idx})", fontsize=12)
plt.tight_layout()
plt.savefig('/tmp/lime_high_risk.png', dpi=150, bbox_inches='tight')
plt.show()

# Patient 2: Low-risk (predicted as No Disease)
low_risk_idx = int(np.where(best_model.predict(X_test) == 0)[0][0])
exp_low = explainer.explain_instance(
    data_row=X_test[low_risk_idx],
    predict_fn=best_model.predict_proba,
    num_features=10
)

print(f"\nPatient {low_risk_idx} — LIME Explanation (Predicted: No Disease):")
print(f"Prediction probability: {best_model.predict_proba(X_test[low_risk_idx:low_risk_idx+1])}")
for feat, weight in exp_low.as_list():
    print(f"  {feat}: {weight:.4f}")

fig = exp_low.as_pyplot_figure()
plt.title(f"LIME Explanation — Low Risk Patient (idx {low_risk_idx})", fontsize=12)
plt.tight_layout()
plt.savefig('/tmp/lime_low_risk.png', dpi=150, bbox_inches='tight')
plt.show()

## LIME vs SHAP: Conclusions

### What LIME Confirmed
- **Consistent top features**: chest pain type (cp), max heart rate (thalach), ST depression (oldpeak), and number of major vessels (ca) appear in both SHAP and LIME explanations, reinforcing their clinical importance.
- **Direction agreement**: For the same patient, LIME and SHAP generally agree on whether a feature pushes toward or away from a positive prediction.

### Key Differences Observed
- **Feature magnitude**: LIME sometimes assigns higher relative importance to binary/categorical features because its local linear approximation treats feature space differently.
- **Stability**: LIME explanations can shift slightly with different perturbation seeds; SHAP is deterministic for tree models.

### When to Use Which?
| Use Case | Recommended Tool |
|----------|-----------------|
| Explaining a single prediction to a doctor | LIME (intuitive, simple rules) |
| Auditing global model fairness | SHAP (consistent, global view) |
| Regulatory compliance (EU AI Act) | Both (triangulation gives more credibility) |
| Model debugging & feature engineering | SHAP (more stable, faster) |

**Bottom Line:** LIME and SHAP are complementary tools. Using both gives a more robust and defensible explanation for high-stakes medical predictions.